# Práctica: Singular Value Decomposition (SVD)

Basado en el **Capítulo 1 de Brunton & Kutz**, *Data-Driven Science and Engineering*. La Descomposición en Valores Singulares (SVD) es la herramienta fundamental en ciencia de datos, equivalente al Análisis de Componentes Principales (PCA).

En esta práctica veremos paso a paso y con bloques de código divididos:
1. **SVD y Aproximación de Matrices** (Compresión de imágenes).
2. **Propiedades Geométricas**.
3. **Pseudo-inversa y Regresión**.
4. **Análisis de Componentes Principales (PCA)**.
5. **Eigenfaces** (Reconocimiento facial y Ejercicio 1).
6. **Truncamiento Óptimo** (Gavish-Donoho).
7. **SVD Aleatoria (rSVD)**.
8. **Introducción a Tensores y SSA**.

## 0. Importaciones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread
from matplotlib.patches import Ellipse
import scipy.integrate as integrate
import scipy.optimize as optimize
from sklearn.datasets import load_sample_image, load_diabetes, load_breast_cancer, fetch_olivetti_faces

# Configuración visual de los gráficos
plt.rcParams.update({
    'figure.figsize': (8, 5),
    'axes.grid': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12
})
np.random.seed(42)

## 1. SVD y Aproximación de Matrices (Sec 1.1 y 1.2)
Cargaremos una imagen y usaremos la SVD económica (reducida) para calcular sus valores singulares.

In [ ]:
china = load_sample_image('china.jpg')
X = china.astype(float).mean(axis=2) / 255.0  # Convertir a escala de grises

print(f"Dimensiones de X: {X.shape}")

plt.imshow(X, cmap='gray')
plt.title('Imagen Original')
plt.axis('off')
plt.show()

Calculamos la SVD económica de `X`. Para una matriz $X \in \mathbb{R}^{n \times m}$ con $n > m$, la SVD económica retorna $U$ de $n \times m$, $S$ de $m$, y $V^T$ de $m \times m$.

In [ ]:
# full_matrices=False indica la SVD económica
U, S, Vt = np.linalg.svd(X, full_matrices=False)

print(f"Dimensiones de U:  {U.shape}")
print(f"Dimensiones de S:  {S.shape} (valores singulares)")
print(f"Dimensiones de Vt: {Vt.shape}")

### Curva de Valores Singulares, Energía y Varianza Explicada
¿Cuántos valores singulares deberíamos conservar? Podemos observarlo graficando el **espectro singular**.
- **Energía Acumulada:** Se calcula sumando los valores singulares $\sigma_k$.
- **Varianza Explicada:** Es proporcional a los cuadrados de los valores singulares $\sigma_k^2$. Representa cuánta información se preserva.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# 1. Espectro (escala logarítmica)
axs[0].semilogy(S, lw=2, color='C0')
axs[0].set_title('Valores Singulares $\sigma_k$')
axs[0].set_xlabel('k'); axs[0].set_ylabel('Magnitud')

# 2. Energía acumulada (Suma de sigma)
energia_acumulada = np.cumsum(S) / np.sum(S)
axs[1].plot(energia_acumulada, lw=2, color='C1')
axs[1].set_title('Energía Acumulada')
axs[1].set_xlabel('k')
axs[1].axhline(0.9, color='gray', linestyle='--', label='90%')
axs[1].legend()

# 3. Varianza explicada acumulada (Suma de sigma^2)
var_acumulada = np.cumsum(S**2) / np.sum(S**2)
axs[2].plot(var_acumulada, lw=2, color='C2')
axs[2].set_title('Varianza Explicada Acumulada')
axs[2].set_xlabel('k')
axs[2].axhline(0.95, color='gray', linestyle='--', label='95%')
axs[2].legend()

plt.tight_layout()
plt.show()

### Compresión de la imagen por truncamiento (Teorema de Eckart-Young)
Calcularemos la reconstrucción de la imagen para distintos valores de $r$ y observaremos cómo afecta a la calidad visual, al porcentaje de almacenamiento y al error.

In [ ]:
r_valores = [5, 20, 50, 100]
fig, axs = plt.subplots(1, len(r_valores), figsize=(18, 5))

for ax, r in zip(axs, r_valores):
    # Reconstrucción truncada: X_approx = U_r * S_r * Vt_r
    X_approx = (U[:, :r] * S[:r]) @ Vt[:r, :]
    
    # Cálculo del error relativo (Norma de Frobenius)
    error = np.linalg.norm(X - X_approx, ord='fro') / np.linalg.norm(X, ord='fro')
    
    # Cálculo del almacenamiento necesario
    almacenamiento = (r * (X.shape[0] + X.shape[1] + 1)) / (X.shape[0] * X.shape[1]) * 100
    
    ax.imshow(np.clip(X_approx, 0, 1), cmap='gray')
    ax.set_title(f'r = {r}\nAlmacenamiento: {almacenamiento:.1f}%\nError Relativo: {error*100:.1f}%')
    ax.axis('off')
    
plt.tight_layout()
plt.show()

## 2. Propiedades y Geometría (Sec 1.3)
La SVD proporciona una transformación geométrica: de una esfera a un elipsoide.

**La propiedad geométrica profunda de la SVD:**
Si consideramos la matriz $X \in \mathbb{R}^{n \times m}$ como una transformación lineal, mapea la esfera unitaria del espacio de dominio $\mathbb{R}^m$ a un hiperelipsoide en $\mathbb{R}^n$.
- Los vectores singulares derechos $v_k$ representan las direcciones principales ortogonales en la esfera original.
- Los vectores singulares izquierdos $u_k$ indican las direcciones de los semiejes principales del elipsoide resultante.
- Los valores singulares $\sigma_k$ son exactamente las longitudes de esos semiejes.
- Por lo tanto, las direcciones de máxima varianza corresponden a los ejes más largos del elipsoide.

In [ ]:
theta = np.linspace(0, 2*np.pi, 100)
circulo = np.vstack((np.cos(theta), np.sin(theta)))

A = np.array([[3, 1], [0.5, 2]])
elipse = A @ circulo

U_A, S_A, Vt_A = np.linalg.svd(A)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(circulo[0], circulo[1], label='Círculo unitario $S^1$')
axs[0].set_aspect('equal')
axs[0].set_title('Espacio de V (dominio)')

axs[1].plot(elipse[0], elipse[1], color='purple')
for j in range(2):
    v_dir = U_A[:, j] * S_A[j]
    axs[1].arrow(0, 0, v_dir[0], v_dir[1], color='red', width=0.05, label=f'$\sigma_{j+1} u_{j+1}$')
axs[1].set_aspect('equal')
axs[1].set_title('Elipse (rango)')
plt.show()

## 3. Pseudo-inversa y Regresión Múltiple (Sec 1.4)
La pseudo-inversa de Moore-Penrose de $X$ es $X^\dagger = V \Sigma^{-1} U^T$. Sirve para resolver el problema de mínimos cuadrados $x = A^\dagger b$.

Vamos a utilizar el dataset **Diabetes** de `scikit-learn` (análogo al dataset Boston Housing clásico).
Contiene variables fisiológicas de pacientes (edad, sexo, masa corporal, presión arterial, etc.) y busca predecir una métrica cuantitativa de la progresión de la enfermedad.

In [ ]:
diabetes = load_diabetes()
print(f"El dataset tiene {diabetes.data.shape[0]} pacientes y {diabetes.data.shape[1]} características.")
print("\nEjemplo - Características del primer paciente:")
print(diabetes.data[0])
print("\nProgresión de la enfermedad (objetivo) del primer paciente:", diabetes.target[0])

Ahora realizamos la regresión multilineal usando la SVD (Calculando la Pseudo-Inversa):

In [ ]:
X_reg = diabetes.data
y_reg = diabetes.target

# Agregar columna de unos para el sesgo (bias)
X_reg = np.c_[X_reg, np.ones(X_reg.shape[0])]

# SVD de la matriz de características
U_reg, S_reg, Vt_reg = np.linalg.svd(X_reg, full_matrices=False)

# Pseudo-inversa y coeficientes
x_tilde = Vt_reg.T @ np.diag(1/S_reg) @ U_reg.T @ y_reg
print(f"Coeficientes calculados (primeros 5): {x_tilde[:5]}")

Evaluemos el error de la predicción y grafiquemos los resultados:

In [ ]:
y_pred = X_reg @ x_tilde
mse = np.mean((y_reg - y_pred)**2)
print(f"Error Cuadrático Medio (MSE): {mse:.2f}")

plt.figure(figsize=(8,5))
plt.plot(y_reg, y_pred, 'o', alpha=0.5)
plt.plot([y_reg.min(), y_reg.max()], [y_reg.min(), y_reg.max()], 'r--', lw=2)
plt.xlabel('Valor Real')
plt.ylabel('Predicción')
plt.title('Regresión Lineal: Predicción vs Realidad')
plt.show()

## 4. PCA (Sec 1.5)
PCA es esencialmente la SVD aplicada a una matriz *centrada por columnas/filas*. Usaremos el dataset **Breast Cancer Wisconsin**.
El dataset agrupa mediciones calculadas a partir de imágenes digitalizadas de masas mamarias. El objetivo es determinar si un tumor es benigno o maligno.

In [ ]:
cancer = load_breast_cancer()
df_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
print(f"El dataset tiene {df_cancer.shape[0]} muestras y {df_cancer.shape[1]} características continuas.")
print("\nEtiquetas posibles:", cancer.target_names)
df_cancer.head()

Procedemos a centrar los datos y extraer las componentes principales mediante la SVD.

In [ ]:
cancer = load_breast_cancer()
X_c = cancer.data
y_c = cancer.target

# Centrar y normalizar (Z-score)
X_c_mean = X_c.mean(axis=0)
X_c_std = X_c.std(axis=0)
X_c_norm = (X_c - X_c_mean) / X_c_std

# SVD (equivalente a PCA)
U_c, S_c, Vt_c = np.linalg.svd(X_c_norm, full_matrices=False)

# Las proyecciones a los componentes principales están dadas por X * V
Z = X_c_norm @ Vt_c.T[:, :2]

plt.scatter(Z[y_c == 0, 0], Z[y_c == 0, 1], label='Maligno', alpha=0.6)
plt.scatter(Z[y_c == 1, 0], Z[y_c == 1, 1], label='Benigno', alpha=0.6)
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('Proyección PCA 2D del dataset Breast Cancer')
plt.legend()
plt.show()

## 5. Eigenfaces (Sec 1.6)
Las **Eigenfaces** (Rostros Propios) son los autovectores (o vectores singulares izquierdos $U$) obtenidos al aplicar SVD a una gran base de datos de rostros humanos.
En lugar de almacenar cada píxel, un rostro se puede describir como una combinación lineal (suma ponderada) del **rostro medio** y de las principales Eigenfaces.
Al usar solo un pequeño número de componentes (las de mayor varianza), podemos comprimir imágenes y extraer los rasgos más importantes para el reconocimiento facial automático.

In [ ]:
faces = fetch_olivetti_faces()
print(f"El dataset tiene {faces.data.shape[0]} imágenes, con 4096 píxeles cada una (64x64).")
print("Vamos a mostrar algunas muestras originales:")

fig, axs = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    axs[i].imshow(faces.data[i].reshape(64, 64), cmap='gray')
    axs[i].axis('off')
plt.suptitle('Muestras Originales')
plt.show()

Cada imagen se vectoriza (se vuelve una sola columna) y restamos el **rostro medio** a todo el conjunto de entrenamiento para centrarlos.

In [ ]:
X_faces = faces.data.T
labels = faces.target

mask_train = labels != 39
mask_test = labels == 39
X_train = X_faces[:, mask_train]
X_test = X_faces[:, mask_test]

rostro_medio = X_train.mean(axis=1, keepdims=True)
B_train = X_train - rostro_medio

U_ef, S_ef, Vt_ef = np.linalg.svd(B_train, full_matrices=False)

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
axs[0].imshow(rostro_medio.reshape(64, 64), cmap='gray')
axs[0].set_title('Rostro Medio'); axs[0].axis('off')
for i in range(1, 4):
    # Los eigenfaces son columnas de U_ef
    axs[i].imshow(U_ef[:, i-1].reshape(64, 64), cmap='gray')
    axs[i].set_title(f'Eigenface {i}')
    axs[i].axis('off')
plt.suptitle('Componentes Base (Eigenfaces)')
plt.show()

### Reconstrucción de un Rostro de Prueba
Vamos a intentar reconstruir un rostro nuevo (no visto en el entrenamiento) sumando el rostro medio más distintas cantidades de *Eigenfaces* $r$. Calcularemos el error relativo (norma Frobenius) respecto a la imagen original (el rostro base).

In [ ]:
rostro_prueba = X_test[:, 0:1]
rostro_centrado = rostro_prueba - rostro_medio

r_valores_ef = [10, 50, 100, 200]
fig, axs = plt.subplots(1, len(r_valores_ef) + 1, figsize=(15, 4))

axs[0].imshow(rostro_prueba.reshape(64, 64), cmap='gray')
axs[0].set_title('Original (Base)')
axs[0].axis('off')

for i, r in enumerate(r_valores_ef):
    # Proyección = U^T * (Rostro - Promedio)
    pesos = U_ef[:, :r].T @ rostro_centrado
    # Reconstrucción = Promedio + U * Pesos
    rostro_reconstruido = rostro_medio + U_ef[:, :r] @ pesos
    
    error_ef = np.linalg.norm(rostro_prueba - rostro_reconstruido) / np.linalg.norm(rostro_prueba)
    
    axs[i+1].imshow(rostro_reconstruido.reshape(64, 64), cmap='gray')
    axs[i+1].set_title(f'r = {r}\nError: {error_ef*100:.1f}%')
    axs[i+1].axis('off')
plt.tight_layout()
plt.show()

### Ejercicio 1 (Guiado): Clasificador de Eigenfaces + 1-NN
Usemos la proyección sobre el subespacio de rostros para reconocer caras.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Separar un test set adecuado (1 imagen de cada persona)
idx_train, idx_test = [], []
for i in range(40):
    idx_train.extend(range(i*10, i*10 + 8)) # 8 fotos por persona
    idx_test.extend(range(i*10 + 8, i*10 + 10)) # 2 fotos por persona

X_tr = X_faces[:, idx_train]
y_tr = labels[idx_train]
X_te = X_faces[:, idx_test]
y_te = labels[idx_test]

rmed = X_tr.mean(axis=1, keepdims=True)
U_tr, S_tr, Vt_tr = np.linalg.svd(X_tr - rmed, full_matrices=False)

rs = [5, 20, 50, 100]
for r in rs:
    # Proyectar sobre r componentes (X^T * U)
    Z_train = (X_tr - rmed).T @ U_tr[:, :r]
    Z_test = (X_te - rmed).T @ U_tr[:, :r]
    
    clf = KNeighborsClassifier(n_neighbors=1)
    clf.fit(Z_train, y_tr)
    acc = accuracy_score(y_te, clf.predict(Z_test))
    print(f"Accuracy con r={r}: {acc*100:.1f}%")

## 7. Truncamiento Óptimo con Ruido Desconocido (Sec 1.7)
¿Qué umbral $\tau$ usar cuando la matriz tiene ruido? Si la magnitud del ruido $\gamma$ es desconocida, Gavish y Donoho proponen:
$$\tau = \omega(\beta) \sigma_{med}$$
donde $\beta = m/n$, $\sigma_{med}$ es la mediana de los valores singulares y $\omega(\beta)$ depende empíricamente de la distribución de Marchenko-Pastur.

In [ ]:
def lambda_beta(beta):
    return np.sqrt(2*(beta+1) + 8*beta / ((beta+1) + np.sqrt(beta**2 + 14*beta + 1)))

def omega_beta(beta):
    # Función aproximada del valor omega de Gavish-Donoho
    # basada en la inversa de la mediana de Marchenko-Pastur
    a = (1 - np.sqrt(beta))**2
    b = (1 + np.sqrt(beta))**2
    f = lambda t: np.sqrt((b-t)*(t-a)) / (2*np.pi*beta*t)
    g = lambda m: integrate.quad(f, a, m)[0] - 0.5
    mu_beta = optimize.brentq(g, a + 1e-9, b - 1e-9)
    return lambda_beta(beta) / np.sqrt(mu_beta)

beta = min(B_train.shape) / max(B_train.shape)
w_beta = omega_beta(beta)
sigma_med = np.median(S_ef)
tau = w_beta * sigma_med

r_optimo = np.sum(S_ef > tau)
print(f"Valor w(beta): {w_beta:.3f}")
print(f"Umbral de truncamiento: {tau:.2f}")
print(f"Rango óptimo r: {r_optimo}")

plt.semilogy(S_ef)
plt.axhline(tau, color='red', linestyle='--', label=f'Umbral $\tau={tau:.1f}$ (r={r_optimo})')
plt.title('Truncamiento Gavish-Donoho para Eigenfaces')
plt.legend()
plt.show()

## 7. SVD Aleatoria - rSVD (Sec 1.8)
Cuando la matriz es gigante, usamos proyecciones aleatorias.

In [ ]:
import time

def rsvd(X, r, q=1, p=5):
    # 1. Proyección aleatoria
    P = np.random.randn(X.shape[1], r + p)
    Z = X @ P
    # Power iterations (estabilizar el rango)
    for k in range(q):
        Z = X @ (X.T @ Z)
    # 2. Base ortonormal
    Q, R = np.linalg.qr(Z, mode='reduced')
    # 3. Proyectar matriz
    Y = Q.T @ X
    # 4. SVD pequeña
    UY, S, Vt = np.linalg.svd(Y, full_matrices=False)
    # 5. Reconstruir U
    U = Q @ UY
    return U, S, Vt

t0 = time.time()
U_det, S_det, Vt_det = np.linalg.svd(X_faces, full_matrices=False)
print(f"Tiempo SVD Determinista: {time.time()-t0:.3f} s")

t0 = time.time()
U_ran, S_ran, Vt_ran = rsvd(X_faces, r=100, q=2)
print(f"Tiempo SVD Aleatoria (r=100): {time.time()-t0:.3f} s")

plt.plot(S_det[:100], label='SVD Determinista')
plt.plot(S_ran[:100], '--', label='rSVD')
plt.legend()
plt.title('Comparación de Valores Singulares')
plt.show()